In [4]:
import pandas as pd
import matplotlib.pyplot as plt

path = "docs/server_logs.csv"
df = pd.read_csv(path)

print("Libreria y data cargada correctamente")

Libreria y data cargada correctamente


### Exploración inicial
- ¿Cuántos logs hay en total?
- ¿Qué severidad aparece más?
- ¿Qué servicio genera más logs?
- ¿Qué servicio genera menos logs?
- ¿Cuál es el mensaje más repetido?
- ¿Cuál es el mensaje “malo” más repetido? (bad events)


In [5]:
print(f"Total de logs: {len(df)}")

if 'severity' in df.columns:
    print(f"\nSeverity mas frecuente:\n{df['severity'].value_counts().head()}")

if 'service_name' in df.columns:
    service_counts = df['service_name'].value_counts()
    print(f"\nServicio con mas Logs: {service_counts.idxmax()} : {service_counts.max()}")
    print(f"Servicio con menos Logs: {service_counts.idxmin()} : {service_counts.min()}")

if 'message' in df.columns:
    print(f"\nMensaje mas repetido:\n{df['message'].value_counts().head(3)}")

if 'message' in df.columns:
    error_severity_mask = df['severity'].isin(['ERROR','CRITICAL'])
    server_error_mask = df['status_code'] >= 500
    bad_event_mask = error_severity_mask | server_error_mask
    df_bad = df[bad_event_mask]
    bad_message_counts = df_bad['message'].value_counts()
    print(f"\nTop 5 Mensajes Malos:\n{bad_message_counts.head(5)}")





Total de logs: 5795

Severity mas frecuente:
severity
INFO        3542
WARN        1358
ERROR        775
CRITICAL     120
Name: count, dtype: int64

Servicio con mas Logs: api-gateway : 1509
Servicio con menos Logs: notification-service : 645

Mensaje mas repetido:
message
Health check OK             1196
Background job completed    1185
Request completed           1161
Name: count, dtype: int64

Top 5 Mensajes Malos:
message
Order creation failed - inventory lock timeout    197
Payment gateway unavailable                       103
Database deadlock detected                         99
Checkout failed - upstream payment timeout         88
Possible credential stuffing detected              69
Name: count, dtype: int64


### Detección del momento crítico
- window_start
- total_events
- bad_events
- bad_rate Seleccionar el momento crítico (top 1 por regla).


In [12]:
df['timestamp_event'] = pd.to_datetime(df['timestamp_event'])

mask_severity = df['severity'].isin(['ERROR','CRITICAL'])
mask_status = df['status_code'] >= 500

df['is_bad'] = mask_severity | mask_status

time_grouped = pd.Grouper(key='timestamp_event', freq='5min')

df_time_windows = (
    df.groupby(time_grouped).agg(
        total_events = ('timestamp_event', 'count'),
        bad_events = ('is_bad', 'sum')
    ).reset_index()
)

df_time_windows['bad_rate'] = (
    df_time_windows['bad_events'] / df_time_windows['total_events']
)

df_windows_filtered = df_time_windows.loc[
    df_time_windows['total_events'] >= 20
]

df_windows_top5 = (df_windows_filtered.sort_values('bad_rate', ascending=False).head(5))
print(f"Top 5:\n{df_windows_top5}")

sr_critical_windows = df_windows_top5.iloc[0]
print(f"\nCritical windows:\n{sr_critical_windows}")

Top 5:
              timestamp_event  total_events  bad_events  bad_rate
134 2026-01-10 11:10:00+00:00           189         110  0.582011
135 2026-01-10 11:15:00+00:00           228         129  0.565789
136 2026-01-10 11:20:00+00:00           111          59  0.531532
463 2026-01-11 14:35:00+00:00           255         117  0.458824
462 2026-01-11 14:30:00+00:00           156          68  0.435897

Critical windows:
timestamp_event    2026-01-10 11:10:00+00:00
total_events                             189
bad_events                               110
bad_rate                            0.582011
Name: 134, dtype: object


### Diagnóstico dentro del momento crítico
Dentro de esa ventana (top 1), generar:
- Tabla: bad events por service_name (ranking)
- Tabla: top 5 message en bad events
- Tabla: top 5 endpoint más comprometidos

Elegí 1 criterio y declaralo explícitamente:
- Por cantidad de bad events
- Por cantidad de status_code >= 500
- Por mayor latency_ms promedio (Nada avanzado. Solo promedio si elegís latencia.)


In [ ]:
dt_windows_start = sr_critical_windows['timestamp_event']
dt_windows_end = dt_windows_start + pd.Timedelta('5min')

mask_windows_start = df['timestamp_event'] >= dt_windows_start
mask_windows_end = df['timestamp_event'] < dt_windows_end

df_critical = df.loc[mask_windows_start & mask_windows_end].copy()

mask_bag_critical = df_critical['is_bad'] == True

df_critical_bad = df_critical.loc[mask_bag_critical].copy()

df_bad_by_service = (
    df_critical_bad.groupby('service_name').agg(n_bad_events = ('is_bad', 'count'))
).reset_index().sort_values('n_bad_events', ascending=False)

print(f"\nTabla: bad events por service_name (ranking):\n{df_bad_by_service}")

KeyError: 'True: boolean label can not be used without a boolean index'